# Basic preprocessed dataset analysis

Inspect one dataset produced by `preprocessing/run_preprocessing.py`. The matrix stays sparse throughout; only small snippets are displayed. Change `DATASET` in the next cell and run all cells.

In [ ]:
from itertools import islice
from pathlib import Path
import gzip

import numpy as np
import pandas as pd
from scipy import sparse

PROJECT = Path("/home/kapachy/Microbium/tensor_feature_selection")
DATASET = "WirbelJ_2018"
SNIPPET_ROWS = 5
SNIPPET_VALUES = 12

data_dir = PROJECT / "outputs" / "preprocessing" / DATASET
required = ["X_valid.npz", "edges.npz", "samples.csv", "filter_summary.csv", "genes.txt.gz", "bacteria.txt.gz"]
missing = [name for name in required if not (data_dir / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing preprocessed files for {DATASET}: {missing}")

matrix = sparse.load_npz(data_dir / "X_valid.npz").tocsr()
samples = pd.read_csv(data_dir / "samples.csv")
filter_summary = pd.read_csv(data_dir / "filter_summary.csv")
with np.load(data_dir / "edges.npz") as stored:
    edges = {name: stored[name] for name in stored.files}

def first_lines(path, count):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return [line.rstrip("\n") for line in islice(handle, count)]

gene_ids_head = first_lines(data_dir / "genes.txt.gz", SNIPPET_ROWS)
bacteria_ids_head = first_lines(data_dir / "bacteria.txt.gz", SNIPPET_ROWS)

assert matrix.shape[0] == len(samples)
assert matrix.shape[1] == len(edges["pair_gene"]) == len(edges["pair_bacteria"])
print(f"Loaded {DATASET}: {matrix.shape[0]:,} samples × {matrix.shape[1]:,} observed gene-bacterium pairs")


## Preprocessing summary and sample metadata

In [ ]:
display(filter_summary.T.rename(columns={0: "value"}))
display(samples.head(SNIPPET_ROWS))
display(
    samples.groupby(["y", "disease"], dropna=False)
    .size().rename("samples").reset_index()
)


## Array snippets

In [ ]:
array_snippets = []
for name, values in edges.items():
    array_snippets.append({
        "array": name,
        "shape": str(values.shape),
        "dtype": str(values.dtype),
        "first values": values[:SNIPPET_VALUES].tolist(),
    })
display(pd.DataFrame(array_snippets))
display(pd.DataFrame({"gene ID snippet": pd.Series(gene_ids_head), "bacteria ID snippet": pd.Series(bacteria_ids_head)}))

matrix_head = matrix[:SNIPPET_ROWS].tocoo()
matrix_snippet = pd.DataFrame({
    "sample_row": matrix_head.row,
    "sample_id": samples.iloc[matrix_head.row]["sample_id"].to_numpy(),
    "pair_column": matrix_head.col,
    "abundance": matrix_head.data,
})
display(matrix_snippet.head(SNIPPET_VALUES))


## Dimensions, storage, and sparsity

In [ ]:
file_sizes = pd.DataFrame([
    {"file": name, "MiB": (data_dir / name).stat().st_size / 2**20}
    for name in required
]).sort_values("MiB", ascending=False)
display(file_sizes.round({"MiB": 2}))

dimensions = pd.Series({
    "samples": matrix.shape[0],
    "pair columns": matrix.shape[1],
    "nonzero values": matrix.nnz,
    "matrix density": matrix.nnz / np.prod(matrix.shape),
    "implicit-zero fraction": 1 - matrix.nnz / np.prod(matrix.shape),
    "mapped bacteria": np.unique(edges["pair_bacteria"]).size,
    "mapped genes": np.unique(edges["pair_gene"]).size,
    "preview-retained pairs": int(np.count_nonzero(edges["global_mask"])),
}, name="value").to_frame()
display(dimensions.map(lambda value: f"{value:,.6g}"))


## Simple distributions

In [ ]:
sample_nonzero = np.diff(matrix.indptr)
sample_mass = np.asarray(matrix.sum(axis=1)).ravel()
pair_sample_support = np.asarray(matrix.getnnz(axis=0)).ravel()
global_sample_support = edges["global_support"]
positive_abundance = matrix.data

distributions = {
    "nonzero pairs per sample": sample_nonzero,
    "total abundance per sample": sample_mass,
    "sample support per pair": pair_sample_support,
    "sample support per pair (preprocessing)": global_sample_support,
    "positive abundance values": positive_abundance,
}
description = pd.DataFrame({
    name: pd.Series(values).describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])
    for name, values in distributions.items()
}).T
display(description.map(lambda value: f"{value:,.4g}"))


## What to check before modeling

- Confirm that labels and sample counts match the intended comparison.
- Compare pair counts before and after the prevalence preview. Evaluation still recomputes this filter inside each training fold.
- Check whether retained mass is close to one; a low value would mean filtering discarded substantial abundance.
- `X_valid` contains the classified pair subset before fold-specific normalization, so its sample totals do not need to equal one.
- Look for extreme sample totals or nonzero counts that could indicate unusual samples.
- Remember that columns are observed gene–bacterium pairs, not a dense bacteria × gene grid.